# MiniLM held-out inference for Cleanlab refits
Attach the saved full-cohort notebook output. This runs inference only using the training-only assessment head. Stop the GPU session once the ZIP is downloaded.

In [ ]:
import sys, subprocess, json, hashlib, shutil, gc
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','-q','pandas>=2.1,<3','transformers>=5.0,<6','safetensors>=0.4'],check=True)
import numpy as np, pandas as pd, torch
assert torch.cuda.is_available(), 'GPU required for held-out inference'
roots=[p for p in Path('/kaggle/input').rglob('car-price-studio-kaggle/training-config.json') if '/notebooks/' in str(p)]
assert len(roots)==1, roots
old=roots[0].parent
root=old.parent.parent
sys.path.insert(0,str(root))
from pricing.text import load_text,predict_text
data=root/'training-cache/full-cohort.jsonl'
assert hashlib.sha256(data.read_bytes()).hexdigest()=='746bc9f8e02e1e708ea5e1d9479408effb131eb6b1d1ea7710380701e62dcf12'
provenance=json.loads((old/'assessment/text-model/provenance.json').read_text())
assert provenance['fitting_rows']==148290, provenance
output=Path('/kaggle/working/cleanlab-text-features');output.mkdir(exist_ok=True)
shutil.copy2(old/'assessment/oof-predictions.csv',output/'assessment-oof.csv')
model,tok=load_text(old/'assessment/text-model','cuda')
torch.set_num_threads(4)
d=pd.concat(pd.read_json(data,lines=True,chunksize=5000),ignore_index=True)
held=d[d.phase!='train'].reset_index(drop=True)
del d;gc.collect()
progress=output/'held-predictions.csv'
completed=set(pd.read_csv(progress).row_id) if progress.exists() else set()
held=held[~held.row_id.isin(completed)]
for start in range(0,len(held),256):
    chunk=held.iloc[start:start+256]
    predictions=predict_text(model,tok,chunk.text.fillna('').tolist(),'cuda',batch_size=8)
    pd.DataFrame({'row_id':chunk.row_id.to_numpy(),'predicted_text_price':predictions}).to_csv(progress,mode='a',header=not progress.exists(),index=False)
    print('HELD-OUT INFERENCE',start+len(chunk),'/',len(held),flush=True)
meta={'cohort_sha256':hashlib.sha256(data.read_bytes()).hexdigest(),'assessment_text_provenance':provenance,'held_rows':63554,'training_oof_rows':148290,'note':'Held-out inference from train-only 148290-row text head; no fine tuning or full-head evaluation.'}
(output/'provenance.json').write_text(json.dumps(meta,indent=2))
del model;gc.collect();torch.cuda.empty_cache()
shutil.make_archive('/kaggle/working/Cleanlab_Text_Features','zip',output)
print('COMPLETE: GPU computation finished; stop session after saving outputs.',flush=True)
